# FRED Stage 2: Qwen Tri-Model Fine-Tuning & Hugging Face Export Pipeline
### Evidence-Aware Financial Hallucination Detection & Selective Editing

This notebook trains all three primary architectures sequentially on the **17,000-sample Stage 2 Dataset** and automatically exports each model to the **Hugging Face Hub** immediately after training:

1. **Model 1: Qwen2.5-3B-Instruct**
   - Fine-tuned with **16-bit LoRA** (unquantized base weights, ~11 GB VRAM peak on Kaggle 16GB T4).
   - In-notebook sanity check across the 5 actions.
   - **Direct Export to Hugging Face Hub**: [`Akash-Purbhi/fredphase2-qwen2.5-3b`](https://huggingface.co/Akash-Purbhi/fredphase2-qwen2.5-3b).
   - GGUF export (`q4_k_m`) for offline deployment.
   - GPU memory cleared before loading the next model.

2. **Model 2: Qwen3-4B (Dense Reasoning / Instruct)**
   - Direct replication of the research paper's core 4B benchmark (`QWEN3-4b-36k*`).
   - Fine-tuned with **16-bit LoRA** (~12 GB VRAM peak on Kaggle 16GB T4).
   - In-notebook sanity check across the 5 actions.
   - **Direct Export to Hugging Face Hub**: [`Akash-Purbhi/fredphase2-qwen3-4b`](https://huggingface.co/Akash-Purbhi/fredphase2-qwen3-4b).
   - GGUF export (`q4_k_m`).
   - GPU memory cleared before loading the next model.

3. **Model 3: Qwen2.5-7B-Instruct**
   - Fine-tuned with **4-bit QLoRA** (~9.5 GB VRAM peak on Kaggle 16GB T4).
   - In-notebook sanity check across the 5 actions.
   - **Direct Export to Hugging Face Hub**: [`Akash-Purbhi/fredphase2-qwen2.5-7b-lora`](https://huggingface.co/Akash-Purbhi/fredphase2-qwen2.5-7b-lora).
   - GGUF export (`q4_k_m`) for Ollama.


## 1. Setup & Environment Installation


In [ ]:
%%capture
# Install Unsloth for fast fine-tuning and minimal memory footprint
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.28" "trl<0.9.0" peft accelerate bitsandbytes datasets huggingface_hub


## 2. Hugging Face Authentication & Hub Target Config
Configure authentication for automatic push to Hub. Reads token from Kaggle Secrets (`HF_TOKEN`), Google Colab `userdata`, or environment variables.


In [ ]:
import os
from huggingface_hub import HfApi, login

# Target Repositories on Hugging Face Hub
HF_REPO_3B = "Akash-Purbhi/fredphase2-qwen2.5-3b"
HF_REPO_4B = "Akash-Purbhi/fredphase2-qwen3-4b"
HF_REPO_7B = "Akash-Purbhi/fredphase2-qwen2.5-7b-lora"

hf_token = None
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    print("Retrieved HF_TOKEN from Kaggle Secrets.")
except Exception:
    pass

if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
        print("Retrieved HF_TOKEN from Colab userdata.")
    except Exception:
        pass

if not hf_token:
    hf_token = os.environ.get("HF_TOKEN")

if not hf_token:
    print("HF_TOKEN not found in secrets/env. If running interactively, enter token below:")
    hf_token = input("Enter Hugging Face Token: ").strip()

if hf_token:
    login(token=hf_token)
    print("Logged in to Hugging Face Hub successfully!")
    print(f"1. Target Repo for 3B: https://huggingface.co/{HF_REPO_3B}")
    print(f"2. Target Repo for 4B: https://huggingface.co/{HF_REPO_4B}")
    print(f"3. Target Repo for 7B: https://huggingface.co/{HF_REPO_7B}")
else:
    print("Warning: No token provided. Models can only be saved locally.")


## 3. Load Stage 2 Datasets
Load `train_chatml.jsonl` (17,000 rows, strictly 50:50 clean:corrupted) and `validation_chatml.jsonl` (1,000 rows).


In [ ]:
import os
import glob
import json
import random
import torch
from pathlib import Path
from datasets import Dataset

SEED = 42
MAX_SEQ_LENGTH = 2048
random.seed(SEED)
torch.manual_seed(SEED)

def find_file(filename):
    candidate_paths = [
        Path(filename),
        Path("data/stage2") / filename,
        Path("../data/stage2") / filename,
    ] + [Path(p) for p in glob.glob(f"/kaggle/input/**/{filename}", recursive=True)] \
      + [Path(p) for p in glob.glob(f"/content/**/{filename}", recursive=True)]
    for p in candidate_paths:
        if p.exists():
            print(f"Found {filename} at: {p}")
            return p
    raise FileNotFoundError(f"Could not find {filename}. Please upload it to your working directory or Kaggle input.")

def load_jsonl(filename):
    path = find_file(filename)
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

train_data = load_jsonl("train_chatml.jsonl")
val_data = load_jsonl("validation_chatml.jsonl")
print(f"Successfully loaded {len(train_data):,} training samples and {len(val_data):,} validation samples.")


---
# 4. Model 1: Qwen2.5-3B-Instruct (16-bit LoRA)
> **Zero Quantization Loss**: On a 16 GB Kaggle T4 GPU, the 3B model base weights require only ~6.2 GB VRAM. Training with unquantized 16-bit LoRA peaks at ~11 GB VRAM, preserving 100% numerical precision.


In [ ]:
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments

print("Loading unsloth/Qwen2.5-3B-Instruct in unquantized 16-bit...")
model_3b, tokenizer_3b = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct",
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=False,  # 3B does not need 4-bit quantization on Kaggle T4!
    token=hf_token,
)
tokenizer_3b.pad_token = tokenizer_3b.eos_token

# Setup LoRA adapters for 3B
model_3b = FastLanguageModel.get_peft_model(
    model_3b,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    max_seq_length=MAX_SEQ_LENGTH,
)

# Format datasets using chat template
def format_3b(batch):
    return {"text": [tokenizer_3b.apply_chat_template(m, tokenize=False, add_generation_prompt=False) for m in batch["messages"]]}

train_ds_3b = Dataset.from_list(train_data).map(format_3b, batched=True)
val_ds_3b = Dataset.from_list(val_data).map(format_3b, batched=True)

training_args_3b = TrainingArguments(
    output_dir="fred_stage2_qwen2_5_3b",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    warmup_ratio=0.03,
    num_train_epochs=2,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=25,
    evaluation_strategy="steps",
    eval_steps=250,
    save_strategy="steps",
    save_steps=500,
    optim="adamw_8bit",
    seed=SEED,
    report_to="none",
)

trainer_3b = SFTTrainer(
    model=model_3b,
    tokenizer=tokenizer_3b,
    train_dataset=train_ds_3b,
    eval_dataset=val_ds_3b,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    args=training_args_3b,
)

print("Starting Qwen2.5-3B training...")
trainer_3b.train()
model_3b.save_pretrained("fred_stage2_qwen2_5_3b_lora")
tokenizer_3b.save_pretrained("fred_stage2_qwen2_5_3b_lora")
print("Qwen2.5-3B fine-tuning completed and saved locally to fred_stage2_qwen2_5_3b_lora!")


### 4.1 In-Notebook Sanity Test: Qwen2.5-3B-Instruct


In [ ]:
FastLanguageModel.for_inference(model_3b)

eval_queries = [
    ("preserve", "Context: Revenue was $500M and net income was $50M.\nQuestion: What was the profit margin?\nResponse to Verify: The profit margin was 10.0%."),
    ("edit", "Context: Revenue was $500M and net income was $50M.\nQuestion: What was the profit margin?\nResponse to Verify: The profit margin was 25.0%."),
    ("abstain", "Context: We manufacture commercial hardware products.\nQuestion: What were litigation reserves in Q3 2023?\nResponse to Verify: Litigation reserves were $100M."),
    ("flag_conflict", "Context: Revenue was $100M.\n[Restatement Note]: Revision shows finalized revenue was $90M.\nQuestion: What was the revenue?\nResponse to Verify: Revenue was $100M."),
    ("request_evidence", "Context: Our corporate mission focuses on sustainability.\nQuestion: What was the effective tax rate?\nResponse to Verify: The effective tax rate was 18%."),
]

print("\n--- Qwen2.5-3B Verification Inference Results ---\n")
for expected_action, query in eval_queries:
    msgs = [
        {"role": "system", "content": "You are FRED Stage 2, an evidence-aware financial hallucination verifier. Given context, tables, a question, and a response, determine the appropriate action: preserve, edit, abstain, flag_conflict, or request_evidence."},
        {"role": "user", "content": query}
    ]
    inputs = tokenizer_3b.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
    outputs = model_3b.generate(inputs, max_new_tokens=256, use_cache=True, temperature=0.1)
    pred = tokenizer_3b.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    print(f"Target Action: {expected_action}\n3B Prediction:\n{pred}\n" + "=" * 50)


### 4.2 Export Qwen2.5-3B to Hugging Face Hub (`Akash-Purbhi/fredphase2-qwen2.5-3b`)


In [ ]:
print(f"Exporting Qwen2.5-3B to Hugging Face Hub: {HF_REPO_3B}...")

if hf_token:
    # Push LoRA adapter and tokenizer directly
    model_3b.push_to_hub(HF_REPO_3B, token=hf_token)
    tokenizer_3b.push_to_hub(HF_REPO_3B, token=hf_token)
    
    # Generate and upload comprehensive model card
    model_card_3b = f"""---
license: apache-2.0
base_model: Qwen/Qwen2.5-3B-Instruct
tags:
- finqa
- tatqa
- hallucination-detection
- financial-nlp
- unsloth
- lora
- qwen2.5
- evidence-aware
datasets:
- FinQA
- TAT-QA
pipeline_tag: text-generation
---

# FRED Stage 2 — Qwen2.5-3B-Instruct (16-bit LoRA)
### Evidence-Aware Financial Hallucination Detection & Selective Editing

This repository contains fine-tuned 16-bit LoRA adapters for **Qwen2.5-3B-Instruct**, trained without 4-bit quantization degradation on the **FRED Stage 2 balanced dataset** (17,000 samples, strictly 50% clean : 50% corrupted).

## Core Capabilities: 5-Action Selective Verification
Rather than blindly regenerating or over-editing text, FRED Stage 2 performs evidence-aware selective verification:
- **`preserve`**: The candidate response is factually grounded and verified against source financial tables/text.
- **`edit`**: Pinpoints numerical or factual hallucinations with in-place XML tags (`<numerical><delete>erroneous</delete><mark>verified</mark></numerical>`).
- **`abstain`**: Context lacks required disclosures to answer without fabricating.
- **`flag_conflict`**: Contradictory statements between table cells and footnotes.
- **`request_evidence`**: Incomplete financial disclosures or missing metrics.

## Model Details
- **Base Model**: `Qwen/Qwen2.5-3B-Instruct`
- **Fine-Tuning Method**: 16-bit LoRA (Unquantized Base, Rank=16, Alpha=16)
- **Training Samples**: 17,000 (8,500 Clean FinQA/TAT-QA + 8,500 11-Error Perturbations)
- **Sequence Length**: 2,048 tokens
- **GitHub Repository**: [FRED GitHub](https://github.com/Akash-Purbhi/FRED)
"""
    with open("README_3b.md", "w", encoding="utf-8") as f:
        f.write(model_card_3b)
        
    api = HfApi(token=hf_token)
    api.upload_file(
        path_or_fileobj="README_3b.md",
        path_in_repo="README.md",
        repo_id=HF_REPO_3B,
    )
    print(f"✅ Qwen2.5-3B Model & README successfully pushed to: https://huggingface.co/{HF_REPO_3B}")
else:
    print("Skipping Hub push (HF_TOKEN not set).")


### 4.3 Export Qwen2.5-3B to GGUF (`q4_k_m`) & Free GPU Memory


In [ ]:
import gc

# Export to 4-bit GGUF for local Ollama use
print("Exporting Qwen2.5-3B to GGUF format (q4_k_m)...")
try:
    model_3b.save_pretrained_gguf(
        "fred-stage2-qwen-3b-q4_k_m",
        tokenizer_3b,
        quantization_method="q4_k_m"
    )
    print("✅ 3B GGUF export complete: fred-stage2-qwen-3b-q4_k_m.gguf")
except Exception as e:
    print(f"GGUF export notice: {e}")

# Free 3B memory before training 4B
print("\nClearing GPU memory before loading Qwen3-4B...")
del model_3b, tokenizer_3b, trainer_3b
gc.collect()
torch.cuda.empty_cache()
print("GPU memory cleared successfully.")


---
# 5. Model 2: Qwen3-4B (Paper Core Architecture)
> **Direct Replication of Paper Benchmark**: The original Tan et al. (2025) study benchmarked `QWEN3-4b-36k*`. Here we fine-tune **Qwen3-4B** in 16-bit LoRA (~12 GB VRAM peak on Kaggle 16GB T4) on the balanced 17,000 Stage 2 dataset.


In [ ]:
print("Loading Qwen3-4B for fine-tuning...")
base_model_name_4b = "Qwen/Qwen3-4B"
try:
    model_4b, tokenizer_4b = FastLanguageModel.from_pretrained(
        model_name=base_model_name_4b,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=False,  # 4B can run in 16-bit on Kaggle T4 (peaks at ~12 GB VRAM)
        token=hf_token,
    )
except Exception as e:
    print(f"Notice: Loading with 4-bit fallback due to: {e}")
    model_4b, tokenizer_4b = FastLanguageModel.from_pretrained(
        model_name=base_model_name_4b,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True,
        token=hf_token,
    )

tokenizer_4b.pad_token = tokenizer_4b.eos_token

model_4b = FastLanguageModel.get_peft_model(
    model_4b,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    max_seq_length=MAX_SEQ_LENGTH,
)

def format_4b(batch):
    return {"text": [tokenizer_4b.apply_chat_template(m, tokenize=False, add_generation_prompt=False) for m in batch["messages"]]}

train_ds_4b = Dataset.from_list(train_data).map(format_4b, batched=True)
val_ds_4b = Dataset.from_list(val_data).map(format_4b, batched=True)

training_args_4b = TrainingArguments(
    output_dir="fred_stage2_qwen3_4b",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    warmup_ratio=0.03,
    num_train_epochs=2,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=25,
    evaluation_strategy="steps",
    eval_steps=250,
    save_strategy="steps",
    save_steps=500,
    optim="adamw_8bit",
    seed=SEED,
    report_to="none",
)

trainer_4b = SFTTrainer(
    model=model_4b,
    tokenizer=tokenizer_4b,
    train_dataset=train_ds_4b,
    eval_dataset=val_ds_4b,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    args=training_args_4b,
)

print("Starting Qwen3-4B training...")
trainer_4b.train()
model_4b.save_pretrained("fred_stage2_qwen3_4b_lora")
tokenizer_4b.save_pretrained("fred_stage2_qwen3_4b_lora")
print("Qwen3-4B fine-tuning completed and saved locally to fred_stage2_qwen3_4b_lora!")


### 5.1 In-Notebook Sanity Test: Qwen3-4B


In [ ]:
FastLanguageModel.for_inference(model_4b)

print("\n--- Qwen3-4B Verification Inference Results ---\n")
for expected_action, query in eval_queries:
    msgs = [
        {"role": "system", "content": "You are FRED Stage 2, an evidence-aware financial hallucination verifier. Given context, tables, a question, and a response, determine the appropriate action: preserve, edit, abstain, flag_conflict, or request_evidence."},
        {"role": "user", "content": query}
    ]
    inputs = tokenizer_4b.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
    outputs = model_4b.generate(inputs, max_new_tokens=256, use_cache=True, temperature=0.1)
    pred = tokenizer_4b.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    print(f"Target Action: {expected_action}\n4B Prediction:\n{pred}\n" + "=" * 50)


### 5.2 Export Qwen3-4B to Hugging Face Hub (`Akash-Purbhi/fredphase2-qwen3-4b`)


In [ ]:
print(f"Exporting Qwen3-4B to Hugging Face Hub: {HF_REPO_4B}...")

if hf_token:
    # Push LoRA adapter and tokenizer directly
    model_4b.push_to_hub(HF_REPO_4B, token=hf_token)
    tokenizer_4b.push_to_hub(HF_REPO_4B, token=hf_token)
    
    # Generate and upload comprehensive model card
    model_card_4b = f"""---
license: apache-2.0
base_model: Qwen/Qwen3-4B
tags:
- finqa
- tatqa
- hallucination-detection
- financial-nlp
- unsloth
- lora
- qwen3
- evidence-aware
datasets:
- FinQA
- TAT-QA
pipeline_tag: text-generation
---

# FRED Stage 2 — Qwen3-4B (Fine-Tuned LoRA)
### Evidence-Aware Financial Hallucination Detection & Selective Editing

This repository contains fine-tuned LoRA adapters for **Qwen3-4B**, replicating the paper's benchmark architecture on the **FRED Stage 2 balanced dataset** (17,000 samples, strictly 50% clean : 50% corrupted).

## Core Capabilities: 5-Action Selective Verification
Rather than blindly regenerating or over-editing text, FRED Stage 2 performs evidence-aware selective verification:
- **`preserve`**: The candidate response is factually grounded and verified against source financial tables/text.
- **`edit`**: Pinpoints numerical or factual hallucinations with in-place XML tags (`<numerical><delete>erroneous</delete><mark>verified</mark></numerical>`).
- **`abstain`**: Context lacks required disclosures to answer without fabricating.
- **`flag_conflict`**: Contradictory statements between table cells and footnotes.
- **`request_evidence`**: Incomplete financial disclosures or missing metrics.

## Model Details
- **Base Model**: `Qwen/Qwen3-4B`
- **Fine-Tuning Method**: LoRA (Rank=16, Alpha=16)
- **Training Samples**: 17,000 (8,500 Clean FinQA/TAT-QA + 8,500 11-Error Perturbations)
- **Sequence Length**: 2,048 tokens
- **GitHub Repository**: [FRED GitHub](https://github.com/Akash-Purbhi/FRED)
"""
    with open("README_4b.md", "w", encoding="utf-8") as f:
        f.write(model_card_4b)
        
    api = HfApi(token=hf_token)
    api.upload_file(
        path_or_fileobj="README_4b.md",
        path_in_repo="README.md",
        repo_id=HF_REPO_4B,
    )
    print(f"✅ Qwen3-4B Model & README successfully pushed to: https://huggingface.co/{HF_REPO_4B}")
else:
    print("Skipping Hub push (HF_TOKEN not set).")


### 5.3 Export Qwen3-4B to GGUF (`q4_k_m`) & Free GPU Memory


In [ ]:
# Export to 4-bit GGUF
print("Exporting Qwen3-4B to GGUF format (q4_k_m)...")
try:
    model_4b.save_pretrained_gguf(
        "fred-stage2-qwen3-4b-q4_k_m",
        tokenizer_4b,
        quantization_method="q4_k_m"
    )
    print("✅ 4B GGUF export complete: fred-stage2-qwen3-4b-q4_k_m.gguf")
except Exception as e:
    print(f"GGUF export notice: {e}")

# Free 4B memory before training 7B
print("\nClearing GPU memory before loading Qwen2.5-7B...")
del model_4b, tokenizer_4b, trainer_4b
gc.collect()
torch.cuda.empty_cache()
print("GPU memory cleared successfully.")


---
# 6. Model 3: Qwen2.5-7B-Instruct (4-bit QLoRA)
> **4-bit Quantization**: The 7B model requires 4-bit quantization (`load_in_4bit=True`) to run within the 16 GB VRAM ceiling of Kaggle T4. Peak VRAM is ~9.5 GB.


In [ ]:
print("Loading unsloth/Qwen2.5-7B-Instruct in 4-bit QLoRA...")
model_7b, tokenizer_7b = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-7B-Instruct",
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,  # 7B strictly requires 4-bit on Kaggle T4
    token=hf_token,
)
tokenizer_7b.pad_token = tokenizer_7b.eos_token

model_7b = FastLanguageModel.get_peft_model(
    model_7b,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    max_seq_length=MAX_SEQ_LENGTH,
)

def format_7b(batch):
    return {"text": [tokenizer_7b.apply_chat_template(m, tokenize=False, add_generation_prompt=False) for m in batch["messages"]]}

train_ds_7b = Dataset.from_list(train_data).map(format_7b, batched=True)
val_ds_7b = Dataset.from_list(val_data).map(format_7b, batched=True)

training_args_7b = TrainingArguments(
    output_dir="fred_stage2_qwen2_5_7b",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    warmup_ratio=0.03,
    num_train_epochs=2,
    learning_rate=1e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=25,
    evaluation_strategy="steps",
    eval_steps=250,
    save_strategy="steps",
    save_steps=500,
    optim="adamw_8bit",
    seed=SEED,
    report_to="none",
)

trainer_7b = SFTTrainer(
    model=model_7b,
    tokenizer=tokenizer_7b,
    train_dataset=train_ds_7b,
    eval_dataset=val_ds_7b,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    args=training_args_7b,
)

print("Starting Qwen2.5-7B training...")
trainer_7b.train()
model_7b.save_pretrained("fred_stage2_qwen2_5_7b_lora")
tokenizer_7b.save_pretrained("fred_stage2_qwen2_5_7b_lora")
print("Qwen2.5-7B fine-tuning completed and saved locally to fred_stage2_qwen2_5_7b_lora!")


### 6.1 In-Notebook Sanity Test: Qwen2.5-7B-Instruct


In [ ]:
FastLanguageModel.for_inference(model_7b)

print("\n--- Qwen2.5-7B Verification Inference Results ---\n")
for expected_action, query in eval_queries:
    msgs = [
        {"role": "system", "content": "You are FRED Stage 2, an evidence-aware financial hallucination verifier. Given context, tables, a question, and a response, determine the appropriate action: preserve, edit, abstain, flag_conflict, or request_evidence."},
        {"role": "user", "content": query}
    ]
    inputs = tokenizer_7b.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
    outputs = model_7b.generate(inputs, max_new_tokens=256, use_cache=True, temperature=0.1)
    pred = tokenizer_7b.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    print(f"Target Action: {expected_action}\n7B Prediction:\n{pred}\n" + "=" * 50)


### 6.2 Export Qwen2.5-7B to Hugging Face Hub (`Akash-Purbhi/fredphase2-qwen2.5-7b-lora`)


In [ ]:
print(f"Exporting Qwen2.5-7B to Hugging Face Hub: {HF_REPO_7B}...")

if hf_token:
    # Push LoRA adapter and tokenizer directly
    model_7b.push_to_hub(HF_REPO_7B, token=hf_token)
    tokenizer_7b.push_to_hub(HF_REPO_7B, token=hf_token)
    
    # Generate and upload comprehensive model card
    model_card_7b = f"""---
license: apache-2.0
base_model: Qwen/Qwen2.5-7B-Instruct
tags:
- finqa
- tatqa
- hallucination-detection
- financial-nlp
- unsloth
- qlora
- qwen2.5
- evidence-aware
datasets:
- FinQA
- TAT-QA
pipeline_tag: text-generation
---

# FRED Stage 2 — Qwen2.5-7B-Instruct (4-bit QLoRA)
### Evidence-Aware Financial Hallucination Detection & Selective Editing

This repository contains fine-tuned 4-bit QLoRA adapters for **Qwen2.5-7B-Instruct**, trained on the **FRED Stage 2 balanced dataset** (17,000 samples, strictly 50% clean : 50% corrupted).

## Core Capabilities: 5-Action Selective Verification
Rather than blindly regenerating or over-editing text, FRED Stage 2 performs evidence-aware selective verification:
- **`preserve`**: The candidate response is factually grounded and verified against source financial tables/text.
- **`edit`**: Pinpoints numerical or factual hallucinations with in-place XML tags (`<numerical><delete>erroneous</delete><mark>verified</mark></numerical>`).
- **`abstain`**: Context lacks required disclosures to answer without fabricating.
- **`flag_conflict`**: Contradictory statements between table cells and footnotes.
- **`request_evidence`**: Incomplete financial disclosures or missing metrics.

## Model Details
- **Base Model**: `Qwen/Qwen2.5-7B-Instruct`
- **Fine-Tuning Method**: 4-bit QLoRA (Rank=16, Alpha=16)
- **Training Samples**: 17,000 (8,500 Clean FinQA/TAT-QA + 8,500 11-Error Perturbations)
- **Sequence Length**: 2,048 tokens
- **GitHub Repository**: [FRED GitHub](https://github.com/Akash-Purbhi/FRED)
"""
    with open("README_7b.md", "w", encoding="utf-8") as f:
        f.write(model_card_7b)
        
    api = HfApi(token=hf_token)
    api.upload_file(
        path_or_fileobj="README_7b.md",
        path_in_repo="README.md",
        repo_id=HF_REPO_7B,
    )
    print(f"✅ Qwen2.5-7B Model & README successfully pushed to: https://huggingface.co/{HF_REPO_7B}")
else:
    print("Skipping Hub push (HF_TOKEN not set).")


### 6.3 Export Qwen2.5-7B to GGUF (`q4_k_m`) for Ollama


In [ ]:
print("Exporting Qwen2.5-7B to GGUF format (q4_k_m)...")
try:
    model_7b.save_pretrained_gguf(
        "fred-stage2-qwen-7b-q4_k_m",
        tokenizer_7b,
        quantization_method="q4_k_m"
    )
    print("✅ 7B GGUF export complete: fred-stage2-qwen-7b-q4_k_m.gguf")
except Exception as e:
    print(f"GGUF export notice: {e}")

print("\n" + "=" * 60)
print("🎉 FRED STAGE 2 TRAINING & EXPORT PIPELINE COMPLETE!")
print(f"1. 3B Model Hub: https://huggingface.co/{HF_REPO_3B}")
print(f"2. 4B Model Hub: https://huggingface.co/{HF_REPO_4B}")
print(f"3. 7B Model Hub: https://huggingface.co/{HF_REPO_7B}")
print("=" * 60)
